[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/valuein/quants/blob/main/examples/notebooks/fundamental_analysis.ipynb)

# Valuein US Core Fundamentals — Financial Analysis

Core fundamental analysis using standardized financial concepts.
Covers revenue trends, balance sheet analysis, and margin ratios.

Every fact in the dataset has two concept columns:
- `concept` — the raw XBRL tag as filed with the SEC (e.g. `us-gaap:Revenues`)
- `standard_concept` — the canonical label our system assigns across all companies
  (e.g. `'Revenues'`), enabling cross-company queries without knowing each filer's
  specific tag choices.

**What you'll learn:**
- How to retrieve income statement items for any ticker
- How to compare companies side by side
- How to compute financial ratios inline with DuckDB SQL
- How `standard_concept` normalization works using `fact.concept` directly

---

In [ ]:
!pip install valuein-sdk -q

In [ ]:
try:
    import os

    from google.colab import userdata

    os.environ["VALUEIN_API_KEY"] = userdata.get("VALUEIN_API_KEY")
except ImportError:
    pass

In [ ]:
from valuein_sdk import ValueinClient

client = ValueinClient(tables=["entity", "security", "filing", "fact"])

## 1. NVIDIA revenue — last 5 annual filings (10-K)

`QUALIFY` deduplicates: each 10-K contains comparative periods; we keep only
the current fiscal year row (latest `period_end` per `fiscal_year`).

In [ ]:
df = client.query("""
    SELECT
        fa.fiscal_year,
        fa.period_end,
        round(fa.numeric_value / 1e9, 2) AS revenue_bn_usd
    FROM   fact    fa
    JOIN   filing  f  ON fa.accession_id = f.accession_id
    JOIN   security s ON f.entity_id     = s.entity_id
    WHERE  s.symbol            = 'NVDA'
      AND  s.is_active         = TRUE
      AND  f.form_type         = '10-K'
      AND  fa.standard_concept = 'TotalRevenue'
      AND  fa.fiscal_period    = 'FY'
    QUALIFY row_number() OVER (
        PARTITION BY fa.fiscal_year ORDER BY fa.period_end DESC
    ) = 1
    ORDER  BY fa.fiscal_year DESC
    LIMIT  5
""")
print(df.to_string(index=False))

## 2. NVIDIA vs Tesla annual revenue comparison

The same `standard_concept = 'TotalRevenue'` filter works across both companies
regardless of which raw XBRL tag each filer used.

In [ ]:
df = client.query("""
    SELECT
        s.symbol,
        fa.fiscal_year,
        round(fa.numeric_value / 1e9, 2) AS revenue_bn_usd
    FROM   fact    fa
    JOIN   filing  f  ON fa.accession_id = f.accession_id
    JOIN   security s ON f.entity_id     = s.entity_id
    WHERE  s.symbol            IN ('NVDA', 'TSLA')
      AND  s.is_active         = TRUE
      AND  f.form_type         = '10-K'
      AND  fa.standard_concept = 'TotalRevenue'
      AND  fa.fiscal_period    = 'FY'
    QUALIFY row_number() OVER (
        PARTITION BY s.symbol, fa.fiscal_year ORDER BY fa.period_end DESC
    ) = 1
    ORDER  BY fa.fiscal_year DESC, s.symbol
    LIMIT  10
""")
print(df.to_string(index=False))

## 3. NVIDIA latest balance sheet snapshot

Use a CTE to find the most recent 10-K filing, then pull the key balance sheet
line items from that single accession.

In [ ]:
df = client.query("""
    WITH latest AS (
        SELECT f.accession_id
        FROM   filing  f
        JOIN   security s ON f.entity_id = s.entity_id
        WHERE  s.symbol    = 'NVDA'
          AND  s.is_active = TRUE
          AND  f.form_type = '10-K'
        ORDER  BY f.filing_date DESC
        LIMIT  1
    )
    SELECT
        fa.standard_concept,
        fa.period_end,
        round(fa.numeric_value / 1e9, 2) AS value_bn_usd
    FROM fact fa
    JOIN latest l ON fa.accession_id = l.accession_id
    WHERE fa.standard_concept IN (
        'TotalAssets',
        'TotalLiabilities',
        'StockholdersEquity'
    )
    QUALIFY row_number() OVER (
        PARTITION BY fa.standard_concept ORDER BY fa.period_end DESC
    ) = 1
    ORDER BY fa.standard_concept
""")
print(df.to_string(index=False))

## 4. Apple gross margin — last 5 annual filings

Join two fact rows from the same filing — one for revenue, one for gross profit —
to compute the gross margin ratio inline in SQL.

In [ ]:
df = client.query("""
    SELECT
        rev.fiscal_year,
        round(rev.numeric_value / 1e9, 2)                          AS revenue_bn,
        round(gp.numeric_value  / 1e9, 2)                          AS gross_profit_bn,
        round(100.0 * gp.numeric_value / rev.numeric_value, 1)     AS gross_margin_pct
    FROM filing f
    JOIN security s  ON f.entity_id    = s.entity_id
    JOIN fact    rev ON rev.accession_id = f.accession_id
                    AND rev.standard_concept = 'TotalRevenue'
                    AND rev.fiscal_period    = 'FY'
    JOIN fact    gp  ON gp.accession_id  = f.accession_id
                    AND gp.standard_concept  = 'GrossProfit'
                    AND gp.fiscal_period     = 'FY'
    WHERE s.symbol    = 'AAPL'
      AND s.is_active = TRUE
      AND f.form_type = '10-K'
    QUALIFY row_number() OVER (PARTITION BY rev.fiscal_year ORDER BY rev.period_end DESC) = 1
    ORDER BY rev.fiscal_year DESC
    LIMIT 5
""")
if df.empty:
    # AAPL uses an unmapped XBRL tag for revenue; fall back to GrossProfit alone
    df = client.query("""
        SELECT
            fa.fiscal_year,
            round(fa.numeric_value / 1e9, 2) AS gross_profit_bn
        FROM fact fa
        JOIN filing  f ON fa.accession_id = f.accession_id
        JOIN security s ON f.entity_id    = s.entity_id
        WHERE s.symbol           = 'AAPL'
          AND s.is_active        = TRUE
          AND f.form_type        = '10-K'
          AND fa.standard_concept = 'GrossProfit'
          AND fa.fiscal_period   = 'FY'
        QUALIFY row_number() OVER (PARTITION BY fa.fiscal_year ORDER BY fa.period_end DESC) = 1
        ORDER BY fa.fiscal_year DESC
        LIMIT 5
    """)
    print("  (TotalRevenue not mapped for AAPL; showing GrossProfit)")
print(df.to_string(index=False))

## 5. Normalization: raw XBRL tags that resolve to 'Revenues'

The `fact` table carries both the raw concept (SEC-filed XBRL tag) and the
`standard_concept` our system assigns. Querying `DISTINCT` pairs shows exactly
how normalization works without needing any private mapping table.

In [ ]:
df = client.query("""
    SELECT DISTINCT
        fa.concept          AS raw_xbrl_tag,
        fa.standard_concept AS normalized_concept
    FROM   fact fa
    WHERE  fa.standard_concept = 'Revenues'
    ORDER  BY fa.concept
    LIMIT  20
""")
print(df.to_string(index=False))
print()
print("One standard_concept. Many raw tags. Normalised across all filers.")